In [ ]:
!nvidia-smi


Fri Jul  3 14:46:30 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import numpy as np
import cupy as cp
import time

# Создаем 20 миллионов точек для симуляции
N = 20_000_000
data_cpu = np.arange(N, dtype=np.float32)

print("--- СЧИТАЕМ НА ПРОЦЕССОРЕ (CPU) ---")
start_time = time.time()
result_cpu = np.square(data_cpu) # Обычный расчет математики
cpu_time = time.time() - start_time
print(f"Время выполнения: {cpu_time:.4f} секунд")

print("\n--- СЧИТАЕМ НА ВИДЕОКАРТЕ (GPU через CUDA) ---")
# Переносим наши данные в видеопамять Tesla T4
data_gpu = cp.asarray(data_cpu)

start_time = time.time()
result_gpu = cp.square(data_gpu)  # Расчет на тысячах ядер CUDA одновременно
cp.cuda.Stream.null.synchronize() # Ждем, пока видеокарта завершит все потоки
gpu_time = time.time() - start_time
print(f"Время выполнения: {gpu_time:.4f} секунд")

print(f"\n🚀 Результат: GPU справился в {cpu_time / gpu_time:.1f} раз(а) быстрее!")


--- СЧИТАЕМ НА ПРОЦЕССОРЕ (CPU) ---
Время выполнения: 0.0241 секунд

--- СЧИТАЕМ НА ВИДЕОКАРТЕ (GPU через CUDA) ---
Время выполнения: 0.0070 секунд

🚀 Результат: GPU справился в 3.4 раз(а) быстрее!


In [ ]:
import numpy as np
import cupy as cp
import time

# Создаем 1 миллион частиц
N_PARTICLES = 1_000_000
STEPS = 100

# Генерируем случайные начальные координаты и скорости
x_init = np.random.rand(N_PARTICLES).astype(np.float32)
y_init = np.random.rand(N_PARTICLES).astype(np.float32)

print("--- СИМУЛЯЦИЯ НА CPU (Процессор) ---")
x = x_init.copy()
y = y_init.copy()

start_time = time.time()
for _ in range(STEPS):
    # Сложный расчет физики движения для каждой частицы
    x = x + np.sin(y) * 0.01
    y = y + np.cos(x) * 0.01
cpu_time = time.time() - start_time
print(f"Время симуляции на CPU: {cpu_time:.4f} секунд")


print("\n--- СИМУЛЯЦИЯ НА GPU (CUDA) ---")
# Переносим координаты частиц в память видеокарты
x_gpu = cp.asarray(x_init)
y_gpu = cp.asarray(y_init)

start_time = time.time()
for _ in range(STEPS):
    # Тот же расчет, но ядра CUDA делают его параллельно для всех частиц сразу
    x_gpu = x_gpu + cp.sin(y_gpu) * 0.01
    y_gpu = y_gpu + cp.cos(x_gpu) * 0.01
cp.cuda.Stream.null.synchronize() # Ждем окончания всех вычислений на GPU
gpu_time = time.time() - start_time
print(f"Время симуляции на GPU: {gpu_time:.4f} секунд")

print(f"\n🚀 Итог на сложной задаче: GPU быстрее в {cpu_time / gpu_time:.1f} раз(а)!")


--- СИМУЛЯЦИЯ НА CPU (Процессор) ---
Время симуляции на CPU: 0.3408 секунд

--- СИМУЛЯЦИЯ НА GPU (CUDA) ---
Время симуляции на GPU: 0.0270 секунд

🚀 Итог на сложной задаче: GPU быстрее в 12.6 раз(а)!


In [ ]:
# 1. Устанавливаем пакеты (werkzeug 3.0.1 решает проблемы совместимости Flask)
!pip install flask==3.0.0 werkzeug==3.0.1 cupy-cuda12x --quiet
!npm install -g localtunnel --quiet

import threading
import urllib.request
from flask import Flask, request, jsonify
import cupy as cp
import numpy as np

app = Flask(__name__)

# Физические параметры расталкивания коробок на развилке
BOX_RADIUS = 0.4
REPU_FORCE = 0.1

@app.route('/api/cuda/simulate', methods=['POST'])
def process_cuda_physics():
    anylogic_data = request.json
    if not anylogic_data:
        return jsonify({"status": "empty"}), 400

    try:
        # 🔥 УМНЫЙ ПАРСИНГ: Извлекаем координаты, даже если AnyLogic прислал их с ключами {'x': ..., 'y': ...}
        extracted_coords = []
        for box in anylogic_data:
            if isinstance(box, dict):
                # Вытаскиваем X и Y, если прилетел JSON-словарь из AnyLogic
                extracted_coords.append([box.get('x', 0.0), box.get('y', 0.0)])
            elif isinstance(box, (list, tuple)) and len(box) >= 2:
                # Если прилетел простой список чисел
                extracted_coords.append([box[0], box[1]])

        if not extracted_coords:
            return jsonify({"status": "success", "updated_positions": []}), 200

        # Превращаем в массив NumPy
        pos_cpu = np.array(extracted_coords, dtype=np.float32)
        num_boxes = len(pos_cpu)

        # 🚀 ЗАГРУЖАЕМ В ПАМЯТЬ GPU ЧЕРЕЗ CUDA
        pos_gpu = cp.asarray(pos_cpu)

        # Вычислительное ядро CUDA (параллельный расчет столкновений на развилке)
        if num_boxes > 1:
            diff = pos_gpu[:, cp.newaxis, :] - pos_gpu[cp.newaxis, :, :]
            dist = cp.sqrt(cp.sum(diff ** 2, axis=-1))
            cp.fill_diagonal(dist, float('inf'))

            collided = dist < (BOX_RADIUS * 2)
            push_direction = diff / (dist[:, :, cp.newaxis] + 1e-5)
            displacement = cp.sum(collided[:, :, cp.newaxis] * push_direction * REPU_FORCE, axis=1)

            # Обновляем координаты коробок по законам физики упругого удара
            pos_gpu += displacement

        # Выгружаем скорректированные физикой координаты обратно на CPU
        new_positions = cp.asnumpy(pos_gpu).tolist()

        print(f"[NVIDIA CUDA Node] Успешно обсчитана физика развилки для {num_boxes} коробок.")
        return jsonify({"status": "success", "updated_positions": new_positions}), 200

    except Exception as e:
        print(f"❌ Ошибка внутри ядра CUDA: {e}")
        return jsonify({"status": "error", "message": str(e)}), 500

# 2. Запускаем Flask в отдельном фоновом потоке на порту 5000!
def run_flask():
    app.run(host='127.0.0.1', port=5000, debug=False, use_reloader=False)

threading.Thread(target=run_flask, daemon=True).start()

# 3. Выводим IP-пароль для localtunnel
print("Ваш IP-адрес для проверки (пароль):", urllib.request.urlopen('https://icanhazip.com').read().decode('utf8').strip())

# 4. Пробрасываем сервер в глобальный интернет по порту 5000!
!lt --port 5000


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.7/226.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.5/133.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.2 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏ * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit


Ваш IP-адрес для проверки (пароль): 8.231.128.185
^C
